# Cloud Native Satellite Data Demo 1
* Use pystac_client to search the STAC API at Planetary Computer
* Use odc.stac to load the data
* Use hvplot to visualize the data

In [1]:
import pystac_client
import planetary_computer
from rich.table import Table
import hvplot.xarray

# Connect to the Planetary Computer STAC API
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

In [2]:
collections = list(catalog.get_all_collections())
collections.sort(key=lambda c: c.id)
table = Table("ID", "Title", title="Planetary Computer collections")
for collection in collections:
    table.add_row(collection.id, collection.title)
table

                                          Planetary Computer collections                                           
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ ID                                                     ┃ Title                                                  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 3dep-lidar-classification                              │ USGS 3DEP Lidar Classification                         │
│ 3dep-lidar-copc                                        │ USGS 3DEP Lidar Point Cloud                            │
│ 3dep-lidar-dsm                                         │ USGS 3DEP Lidar Digital Surface Model                  │
│ 3dep-lidar-dtm                                         │ USGS 3DEP Lidar Digital Terrain Model                  │
│ 3dep-lidar-dtm-native                                  │ USGS 3DEP Lidar Digital Terrain Model (Native)         │
│ 3dep-lidar-hag                                         │ USGS 3DEP Lidar Height above Ground                    │
│ 3dep-lidar-intensity                                   │ USGS 3DEP Lidar Intensity                              │
│ 3dep-lidar-pointsourceid                               │ USGS 3DEP Lidar Point Source                           │
│ 3dep-lidar-returns                                     │ USGS 3DEP Lidar Returns                                │
│ 3dep-seamless                                          │ USGS 3DEP Seamless DEMs                                │
│ alos-dem                                               │ ALOS World 3D-30m                                      │
│ alos-fnf-mosaic                                        │ ALOS Forest/Non-Forest Annual Mosaic                   │
│ alos-palsar-mosaic                                     │ ALOS PALSAR Annual Mosaic                              │
│ aster-l1t                                              │ ASTER L1T                                              │
│ chesapeake-lc-13                                       │ Chesapeake Land Cover (13-class)                       │
│ chesapeake-lc-7                                        │ Chesapeake Land Cover (7-class)                        │
│ chesapeake-lu                                          │ Chesapeake Land Use                                    │
│ chloris-biomass                                        │ Chloris Biomass                                        │
│ cil-gdpcir-cc-by                                       │ CIL Global Downscaled Projections for Climate Impacts  │
│                                                        │ Research (CC-BY-4.0)                                   │
│ cil-gdpcir-cc-by-sa                                    │ CIL Global Downscaled Projections for Climate Impacts  │
│                                                        │ Research (CC-BY-SA-4.0)                                │
│ cil-gdpcir-cc0                                         │ CIL Global Downscaled Projections for Climate Impacts  │
│                                                        │ Research (CC0-1.0)                                     │
│ conus404                                               │ CONUS404                                               │
│ cop-dem-glo-30                                         │ Copernicus DEM GLO-30                                  │
│ cop-dem-glo-90                                         │ Copernicus DEM GLO-90                                  │
│ daymet-annual-hi                                       │ Daymet Annual Hawaii                                   │
│ daymet-annual-na                                       │ Daymet Annual North America                            │
│ daymet-annual-pr                                       │ Daymet Annual Puerto Rico                              │
│ daymet-daily-hi                                       

In [3]:
# Define our area of interest (AOI) - a rough box around Cape Cod gulf of tunisia
bbox = [10.0, 36.6, 10.9, 37.2]

In [4]:
# Let's search for Sentinel-2 Level-2A data from this past summer NADIA
# with minimal cloud cover.
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2026-01-01/2026-04-22",
    query={"eo:cloud_cover": {"lt": 10}}, # less than 10% clouds
)

# Get the results and see what we found
items = search.item_collection()
print(f"Found {len(items)} scenes matching your criteria.")

# Let's inspect the assets of the first item to see the available data bands
if items:
    first_item = items[0]
    print("\nAssets for the first scene:")
    for asset_key, asset in first_item.assets.items():
        print(f"- {asset_key}: {asset.title}")

Found 43 scenes matching your criteria.

Assets for the first scene:
- AOT: Aerosol optical thickness (AOT)
- B01: Band 1 - Coastal aerosol - 60m
- B02: Band 2 - Blue - 10m
- B03: Band 3 - Green - 10m
- B04: Band 4 - Red - 10m
- B05: Band 5 - Vegetation red edge 1 - 20m
- B06: Band 6 - Vegetation red edge 2 - 20m
- B07: Band 7 - Vegetation red edge 3 - 20m
- B08: Band 8 - NIR - 10m
- B09: Band 9 - Water vapor - 60m
- B11: Band 11 - SWIR (1.6) - 20m
- B12: Band 12 - SWIR (2.2) - 20m
- B8A: Band 8A - Vegetation red edge 4 - 20m
- SCL: Scene classfication map (SCL)
- WVP: Water vapour (WVP)
- visual: True color image
- safe-manifest: SAFE manifest
- granule-metadata: Granule metadata
- inspire-metadata: INSPIRE metadata
- product-metadata: Product metadata
- datastrip-metadata: Datastrip metadata
- tilejson: TileJSON with default rendering
- rendered_preview: Rendered preview


In [5]:
# Let's search for Sentinel-3 SLSTR WST (Sea Surface Temperature) Level-2 data  NADIA
search = catalog.search(
    collections=["sentinel-3-slstr-wst-l2-netcdf"],
    bbox=bbox,
    datetime="2026-01-01/2026-04-21",
)

# Get the results and see what we found
items = search.item_collection()
print(f"Found {len(items)} scenes matching your criteria.")

# Let's inspect the assets of the first item to see the available data bands
if items:
    first_item = items[0]
    print("\nAssets for the first scene:")
    for asset_key, asset in first_item.assets.items():
        print(f"- {asset_key}: {asset.title}")

Found 302 scenes matching your criteria.

Assets for the first scene:
- l2p: None
- browse-jpg: None
- eop-metadata: None
- safe-manifest: None


In [5]:
# Assuming 'items' is the ItemCollection from the first step
from odc.stac import stac_load


# Load the data using odc-stac.
# Note that we specify the dask chunks as a dictionary.
data_ds = stac_load(
    items,
    bands=["B04", "B03", "B02", "B08"], # B04=Red, B03=Green, B02=Blue, B08=NIR
    bbox=bbox,
    resolution=10,
    chunks={"x": 2048, "y": 2048},
)

# Let's inspect our new xarray Dataset.
# Notice the "Data variables" section.
print(data_ds)

ValueError: No such band/alias: B04

In [6]:
import xarray as xr
import fsspec

# Prendre le premier item
item = items[0]

# Signer l'URL
asset = item.assets["l2p"]
signed_asset = planetary_computer.sign(asset)

print(f"URL : {signed_asset.href}")

# Ouvrir avec xarray
ds = xr.open_dataset(fsspec.open(signed_asset.href).open())
print(ds)

URL : https://sentinel3euwest.blob.core.windows.net/sentinel-3/SLSTR/SL_2_WST___/2026/04/16/S3A_SL_2_WST____20260416T081600_20260416T095659_20260417T140055_6059_138_206______MAR_O_NT_003.SEN3/20260416081600-MAR-L2P_GHRSST-SSTskin-SLSTRA-20260417140055-v02.0-fv01.0.nc?st=2026-04-21T09%3A51%3A34Z&se=2026-04-22T10%3A36%3A34Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2026-04-20T20%3A27%3A33Z&ske=2026-04-27T20%3A27%3A33Z&sks=b&skv=2025-07-05&sig=ZKnIgXKttux95kGtxx9CnC69n/bTuh2lwBQb3ARKZSk%3D


/srv/conda/envs/notebook/lib/python3.13/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: Engine 'kerchunk' loading failed:
No module named 'zarr.core.array_spec'; 'zarr.core' is not a package
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
/tmp/ipykernel_25770/3867745640.py:14: FutureWarning: In a future version, xarray will not decode the variable 'sst_dtime' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  ds = xr.open_dataset(fsspec.open(signed_asset.href).open())


<xarray.Dataset> Size: 12GB
Dimensions:                            (time: 1, nj: 40394, ni: 1500, channel: 3)
Coordinates:
    lat                                (nj, ni) float32 242MB ...
    lon                                (nj, ni) float32 242MB ...
  * time                               (time) datetime64[ns] 8B 2026-04-16T08...
Dimensions without coordinates: nj, ni, channel
Data variables: (12/22)
    adi_dtime_from_sst                 (time, nj, ni) float64 485MB ...
    aerosol_dynamic_indicator          (time, nj, ni) float64 485MB ...
    brightness_temperature             (channel, time, nj, ni) float64 1GB ...
    dt_analysis                        (time, nj, ni) float64 485MB ...
    dual_nadir_sst_difference          (time, nj, ni) float64 485MB ...
    l2p_flags                          (time, nj, ni) int16 121MB ...
    ...                                 ...
    sses_standard_deviation            (time, nj, ni) float64 485MB ...
    sst_algorithm_type                 

In [7]:
import numpy as np

# Charger seulement lat/lon d'abord (légères)
lat = ds["lat"].values
lon = ds["lon"].values

# Masque spatial : golfe de Tunis
mask = (
    (lat >= 36.6) & (lat <= 37.2) &
    (lon >= 10.0) & (lon <= 10.9)
)

# Indices valides
nj_idx, ni_idx = np.where(mask)

if len(nj_idx) == 0:
    print("❌ Aucun pixel dans la bbox — essaie un autre item")
else:
    print(f"✅ {len(nj_idx)} pixels trouvés dans le golfe de Tunis")

    # Slice les indices min/max pour extraire un bloc rectangulaire
    nj_min, nj_max = nj_idx.min(), nj_idx.max()
    ni_min, ni_max = ni_idx.min(), ni_idx.max()

    # Extraire la SST sur cette zone seulement
    sst_subset = ds["sea_surface_temperature"].isel(
        nj=slice(nj_min, nj_max),
        ni=slice(ni_min, ni_max),
        time=0
    ).load()

    lat_subset = ds["lat"].isel(
        nj=slice(nj_min, nj_max),
        ni=slice(ni_min, ni_max)
    ).values

    lon_subset = ds["lon"].isel(
        nj=slice(nj_min, nj_max),
        ni=slice(ni_min, ni_max)
    ).values

    # Convertir Kelvin → Celsius
    sst_celsius = sst_subset - 273.15

    print(f"SST min: {float(sst_celsius.min()):.1f}°C")
    print(f"SST max: {float(sst_celsius.max()):.1f}°C")

✅ 5377 pixels trouvés dans le golfe de Tunis
SST min: -0.5°C
SST max: 20.4°C


In [8]:
import numpy as np
import xarray as xr
import hvplot.pandas
import pandas as pd



# Filtrer les valeurs invalides (< 10°C = terre ou nuage pour la Méditerranée)
sst_vals = sst_celsius.values.flatten()
lat_vals = lat_subset.flatten()
lon_vals = lon_subset.flatten()

# Créer un DataFrame et filtrer
df = pd.DataFrame({
    "lat": lat_vals,
    "lon": lon_vals,
    "sst": sst_vals
})

# Garder uniquement les pixels valides
df = df[(df["sst"] > 10) & (df["sst"] < 40)]

print(f"Pixels valides : {len(df)}")
print(f"SST min: {df['sst'].min():.1f}°C")
print(f"SST max: {df['sst'].max():.1f}°C")
print(f"SST moyenne: {df['sst'].mean():.1f}°C")

# Visualisation hvplot

plot = df.hvplot.points(
    x="lon", y="lat",
    c="sst",
    colormap="RdYlBu_r",
    title="Sea Surface Temperature — Gulf of Tunisia ",
    xlabel="Longitude",
    ylabel="Latitude",
    clabel="SST (°C)",
    size=8,
    geo=True,
    tiles="OSM",
    width=700,
    height=500
)

plot  # ← suffit dans Jupyter, pas besoin de hvplot.show()

Pixels valides : 1412
SST min: 10.0°C
SST max: 20.4°C
SST moyenne: 13.4°C


:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [lon,lat]   (sst)

In [ ]:
import hvplot.xarray  # Make sure to import this to activate the .hvplot accessor
import numpy as np

# --- Step 1: Find the clearest scene (same as before) ---
print("Finding the clearest available scene...")
valid_pixels = data_ds.B02.where(data_ds.B02 > 0).count(dim=["x", "y"])
best_time_slice = valid_pixels.argmax().compute()
best_scene = data_ds.isel(time=best_time_slice)
scene_date = np.datetime_as_string(best_scene.time.values, unit='D')
print(f"-> Found scene from: {scene_date}")

# --- Step 2: Prepare the RGB data (same as before) ---
# The order B04, B03, B02 corresponds to Red, Green, Blue
rgb = best_scene[["B04", "B03", "B02"]].to_array(dim="band")

# --- Step 3: Scale the data for visualization (same as before) ---
rgb_scaled = rgb.clip(0, 3000) / 3000

In [ ]:
source_crs = f"{items[0].properties['proj:code']}"
source_crs

In [ ]:
# --- Step 4: Plot with hvplot! ---
print("Generating interactive plot...")
# The .hvplot.rgb method is specifically designed for this task.
# The 'bands' argument tells it which dimension holds the R, G, B channels.
image_plot = rgb_scaled.hvplot.rgb(
    x='x',
    y='y',
    bands='band',
    rasterize=True,    # Essential for good performance with large images
    frame_width=600,
    crs=source_crs,
    geo=True,
    flip_yaxis=False,   # not needed as hvplot figures this out
    title=f"{scene_date}"
)

# In a Jupyter notebook, this will display the interactive plot automatically
image_plot